# Calling the keyword segmenter endpoint

`keywords_extracted` in the published table comes from one model service: a
fine-tuned `deepset/gbert-base` that splits the upstream `keywords` string into
single keywords. It runs on Modal, CPU-only, with `min_containers=0` -- so the first
request after a quiet period pays ~20s of cold start.

| | |
| --- | --- |
| URL | `FDB_TAGGER_URL`, the deployed `.modal.run` endpoint |
| Auth | `X-Tagger-Token: $FDB_TAGGER_TOKEN`. **Not** `Authorization` -- Modal's web-endpoint proxy reserves that header and the value never reaches the handler |
| Request | `{"keywords": ["...", ...]}`, at most 512 values |
| Response | `{"model": "...", "results": [{"terms": [...], "group_sizes": [...]}, ...]}`, in request order |

`group_sizes` is what the model actually predicts: one size per keyword, summing to
the number of whitespace tokens in the input. `terms` is that partition applied to
the string -- which is why every keyword is a contiguous span of the input and
invention, omission and reordering are unrepresentable rather than merely untested.

The model, the measurements and the deployment:
[services/keyword_segmenter/README.md](../services/keyword_segmenter/README.md).

In [3]:
import os
from pathlib import Path

# The endpoint's URL and token live in a gitignored .env.tagger at the repo root --
# the URL is the deployed Modal endpoint, the token is the fdb-tagger-token secret.
# Read here rather than requiring `set -a; . ./.env.tagger` before jupyter started.
env = Path.cwd().parent / ".env.tagger"
if env.is_file():
    for line in env.read_text().splitlines():
        line = line.removeprefix("export ").strip()
        if line and not line.startswith("#"):
            key, _, value = line.partition("=")
            os.environ.setdefault(key, value)

missing = [v for v in ("FDB_TAGGER_URL", "FDB_TAGGER_TOKEN") if not os.environ.get(v)]
if missing:
    raise SystemExit(
        f"not set, and not in {env}: {', '.join(missing)}. "
        "See services/keyword_segmenter/README.md > Usage."
    )

URL = os.environ["FDB_TAGGER_URL"]
TOKEN = os.environ["FDB_TAGGER_TOKEN"]
# Neither is echoed. A saved output would commit the endpoint URL into the repo,
# and the URL is worth keeping quiet: auth is checked inside the handler, so an
# unauthorised request has already paid for a container start. See tagger_app.py.
f"loaded URL and token from {env.name}"

'loaded URL and token from .env.tagger'

## One POST

The second value is the hard case from the README: eight tokens joined by spaces
alone. Note that the model does *not* reproduce the gold labelling here -- it returns
`Beauftragte der Bundesregierung für` / `Kultur und Medien` / `BKM` where the label is
one keyword. Boundary placement is exactly the part the held-out F1 measures, and
0.967 is not 1.0. What the span invariant guarantees is the rest: no keyword is
invented, dropped, reordered or reworded.

In [4]:
import requests

values = [
    "Erneuerbare Energien Zuschuss Kommune",
    "Pflege polnischer Sprache Beauftragte der Bundesregierung für Kultur und Medien BKM",
]

response = requests.post(
    URL,
    json={"keywords": values},
    headers={"x-tagger-token": TOKEN},
    timeout=600,  # generous: a cold container loads 437 MB of weights first
)
response.raise_for_status()
body = response.json()

print(body["model"])
for value, result in zip(values, body["results"]):
    print(f"\n{value}\n  -> {result['terms']}\n     sizes={result['group_sizes']}")

deepset/gbert-base

Erneuerbare Energien Zuschuss Kommune
  -> ['Erneuerbare Energien', 'Zuschuss', 'Kommune']
     sizes=[2, 1, 1]

Pflege polnischer Sprache Beauftragte der Bundesregierung für Kultur und Medien BKM
  -> ['Pflege', 'polnischer Sprache', 'Beauftragte der Bundesregierung für', 'Kultur und Medien', 'BKM']
     sizes=[1, 2, 4, 3, 1]


## The same call as the pipeline makes it

`TaggerClient` adds what a batch run needs: deduplication, chunking at 256 values,
retries on a cold start or a 5xx, and a read-through sqlite cache keyed on
`md5(keywords)`. A rerun costs nothing and only genuinely new strings hit the
network.

It is what `fdb_scraper.history.segment_keywords` calls. Its results are materialised
in the `keyword_segments` table, and `publish()` reads the column from there rather
than re-running the model -- inference is not bit-reproducible, so two publishes of
one history would otherwise differ.

In [5]:
import sys
from pathlib import Path

# Not an installed package: the service lives beside its training data and Modal app.
sys.path.insert(0, str(Path.cwd().parent / "services" / "keyword_segmenter"))
from segment.client import Cache, TaggerClient  # noqa: E402

with TaggerClient(cache=Cache("/tmp/keyword_demo.sqlite")) as client:
    fresh = client.segment(values)
    again = client.segment(values)  # served from sqlite, no request

assert fresh == again
fresh

[{'terms': ['Erneuerbare Energien', 'Zuschuss', 'Kommune'],
  'group_sizes': [2, 1, 1]},
 {'terms': ['Pflege',
   'polnischer Sprache',
   'Beauftragte der Bundesregierung für',
   'Kultur und Medien',
   'BKM'],
  'group_sizes': [1, 2, 4, 3, 1]}]